# 1. Import necessary packages

In [40]:
import os

import turmoric
from turmoric.image_process import nd2_to_tif
from turmoric.utils import recursively_get_all_filepaths
from turmoric.apply_thresholds import apply_all_thresh

from pathlib import Path

import pandas as pd
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
from pathlib import Path
from skimage import filters, io
from skimage.segmentation import clear_border
from skimage.morphology import remove_small_objects
from skimage.measure import block_reduce, label, regionprops, regionprops_table
from skimage import exposure
from scipy import ndimage

from turmoric.cell_analysis import apply_regionprops_recursively
from turmoric.utils import organize_files_without_leakage
from turmoric.vampire_model import VampireModelTrainer
from turmoric.split_functions import *
from turmoric.grouping_strategy import CustomFunctionStrategy


from PIL import Image

from sklearn.model_selection import train_test_split

import sklearn
import shutil
from typing import List, Dict, Callable, Optional, Any


import vampire
from vampire.plot import *

import matplotlib.pyplot as plt

import warnings
import logging

# Suppress matplotlib font warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)


import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold

import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import StandardScaler
from pathlib import Path


In [ ]:
all_tifs = recursively_get_all_filepaths("/gscratch/cheme/nlsschim/data/cross_species/tifs/specific_studies/split_adult_wm_gm", file_type=".tif")
print(len(all_tifs))

In [7]:
props_list = ('area', 'bbox_area', 'centroid', 'convex_area',
              'eccentricity', 'equivalent_diameter', 'euler_number',
              'extent', 'filled_area', 'major_axis_length',
              'minor_axis_length', 'orientation', 'perimeter', 'solidity',
              'feret_diameter_max')

In [8]:
regionprops_dfs = []

for file in all_tifs:
    im = tiff.imread(file)

    label_im = label(im)
    props = regionprops_table(label_im, properties=props_list)

    props_df = pd.DataFrame(props)
    props_df['filename'] = file

    species = file.split("/")[9]

    props_df['species'] = species
    regionprops_dfs.append(props_df)

all_regionprops_data = pd.concat(regionprops_dfs)

In [ ]:
# regionprops_df = apply_regionprops_recursively(input_folder="/gscratch/cheme/nlsschim/data/cross_species/tifs/all_data/", 
#                                                properties_list=props_list)
all_regionprops_data['circularity'] = 4*np.pi*all_regionprops_data.area/all_regionprops_data.perimeter**2
all_regionprops_data['aspect_ratio'] = all_regionprops_data.major_axis_length/all_regionprops_data.minor_axis_length
all_regionprops_data

In [ ]:
medians = all_regionprops_data[["area", "bbox_area", "convex_area", "perimeter", "extent", "solidity", "circularity", "aspect_ratio", "eccentricity"]].median()
medians

In [12]:
all_regionprops_data.to_csv("split_adult_wm_gm.csv", index=False)

In [13]:
all_regionprops_data = pd.read_csv("split_adult_wm_gm.csv")

In [14]:
morphology_features = ['area', 'bbox_area', 'convex_area',
              'eccentricity', 'equivalent_diameter', 'euler_number',
              'extent', 'filled_area', 'major_axis_length',
              'minor_axis_length', 'orientation', 'perimeter', 'solidity',
              'feret_diameter_max', 'circularity', 'aspect_ratio']

In [14]:
# Define your base directory and target directories for training and testing
base_dir = "/gscratch/cheme/nlsschim/data/cross_species/tifs/specific_studies/adult_wm_gm"
train_dir = "/gscratch/cheme/nlsschim/data/cross_species/tifs/specific_studies/adult_wm_gm/training"
test_dir = "/gscratch/cheme/nlsschim/data/cross_species/tifs/specific_studies/adult_wm_gm/testing"

# Define a list of subfolder names or patterns to look for
treatment_conditions = ["cortex", "corpus_col"]
groups = ["human_female", "human_male",  "mice_female",  "mice_male",  "pig_female",  "rat_female",  "rat_male"]
#groups = ["human_female"]

# Create training and testing directories if they don't exist
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

In [ ]:
image_set_path = train_dir

train_properties_df = vampire.extraction.extract_properties(img_set_path=image_set_path)

In [6]:
build_info_df = pd.DataFrame({
    'img_set_path': [image_set_path],
    'output_path': [image_set_path],
    'model_name': ['li'],
    'num_points': [50],
    'num_clusters': [5],
    'num_pc': [np.nan]
})

In [ ]:
vampire.quickstart.fit_models(build_info_df)

In [8]:
model_path = os.path.join(image_set_path, 'model_li_(50_5_25)__.pickle')
vampire_model = vampire.util.read_pickle(model_path)

In [16]:
groups = ["human_female", "human_male",  "mice_female",  "mice_male",  "pig_female",  "rat_female",  "rat_male"]

In [16]:
from pathlib import Path
from collections import defaultdict
import random
import shutil
import tempfile
from typing import Optional

def create_stratified_kfolds(
    base_dir: str,
    groups: list[str],
    treatment_conditions: list[str],
    k: int = 5,
    file_type: str = ".tif",
    random_seed: int = 42
) -> list[list[Path]]:
    """
    Create k stratified folds from microscopy images, ensuring equal representation
    across species/sex groups and treatment conditions.

    Stratification works by finding the minimum number of images available across
    all group/condition combinations, then sampling that many from each stratum
    to guarantee balanced folds.

    Parameters
    ----------
    base_dir : str
        Root directory structured as base_dir/group/condition/*.tif
    groups : list of str
        Species/sex groups (e.g. ["human_female", "mice_male"])
    treatment_conditions : list of str
        Brain region conditions (e.g. ["cortex", "corpus_col"])
    k : int, optional
        Number of folds. Default is 5.
    file_type : str, optional
        File extension to look for. Default is ".tif".
    random_seed : int, optional
        Random seed for reproducibility. Default is 42.

    Returns
    -------
    folds : list of list of Path
        A list of k folds, where each fold is a list of file paths.
        Each fold has equal representation from all group/condition strata.

    Notes
    -----
    - Files are sampled without replacement within each stratum.
    - Strata with fewer than k files are skipped with a warning.
    - The number of files per fold per stratum = floor(stratum_size / k).
    - Total files used = n_valid_strata * k * files_per_fold_per_stratum.

    Examples
    --------
    >>> folds = create_stratified_kfolds(
    ...     base_dir="/data/cross_species/tifs/adult_wm_gm",
    ...     groups=["human_female", "mice_male", "rat_female"],
    ...     treatment_conditions=["cortex", "corpus_col"],
    ...     k=5
    ... )
    >>> print(f"Fold sizes: {[len(f) for f in folds]}")
    """
    random.seed(random_seed)
    base_path = Path(base_dir)

    # --- Collect files per stratum ---
    strata: dict[tuple[str, str], list[Path]] = defaultdict(list)

    for group in groups:
        for condition in treatment_conditions:
            stratum_path = base_path / group / condition
            if not stratum_path.exists():
                print(f"  [WARNING] Path not found, skipping: {stratum_path}")
                continue

            files = sorted(stratum_path.glob(f"*{file_type}"))
            if len(files) < k:
                print(
                    f"  [WARNING] {group}/{condition} has only {len(files)} files "
                    f"(need >= {k}). Skipping this stratum."
                )
                continue

            strata[(group, condition)] = files
            print(f"  {group}/{condition}: {len(files)} files")

    if not strata:
        raise ValueError("No valid strata found. Check base_dir and group/condition names.")

    # --- Find the minimum per-fold contribution across strata ---
    # Each stratum contributes floor(n_files / k) files total, split evenly across folds
    files_per_fold_per_stratum = min(len(files) // k for files in strata.values())
    total_per_stratum = files_per_fold_per_stratum * k

    print(f"\nFiles per fold per stratum : {files_per_fold_per_stratum}")
    print(f"Total files used per stratum: {total_per_stratum}")
    print(f"Total strata               : {len(strata)}")
    print(f"Total files across all folds: {total_per_stratum * len(strata)}")

    # --- Build k balanced folds ---
    folds: list[list[Path]] = [[] for _ in range(k)]

    for (group, condition), files in strata.items():
        # Randomly sample exactly total_per_stratum files from this stratum
        sampled = random.sample(files, total_per_stratum)

        # Distribute evenly: fold i gets sampled[i::k] (round-robin assignment)
        for fold_idx in range(k):
            fold_files = sampled[fold_idx * files_per_fold_per_stratum :
                                 (fold_idx + 1) * files_per_fold_per_stratum]
            folds[fold_idx].extend(fold_files)

    # Shuffle within each fold so files aren't ordered by stratum
    for fold in folds:
        random.shuffle(fold)

    print(f"\nFold sizes: {[len(f) for f in folds]}")
    return folds


def kfold_to_temp_dirs(
    folds: list[list[Path]],
    fold_idx: int,
    prefix: str = "vampire_fold"
) -> tuple[Path, Path, Path]:
    """
    Copy files for a single fold into temporary train/test directories.

    Parameters
    ----------
    folds : list of list of Path
        Output of create_stratified_kfolds.
    fold_idx : int
        Index of the fold to use as the test set.
    prefix : str, optional
        Prefix for the temporary directory name.

    Returns
    -------
    temp_dir : Path
        Root temporary directory (caller is responsible for cleanup).
    train_dir : Path
        Directory containing training files.
    test_dir : Path
        Directory containing test files.

    Examples
    --------
    >>> folds = create_stratified_kfolds(base_dir, groups, conditions)
    >>> for fold_idx in range(len(folds)):
    ...     temp_dir, train_dir, test_dir = kfold_to_temp_dirs(folds, fold_idx)
    ...     try:
    ...         # run vampire training/testing here
    ...         pass
    ...     finally:
    ...         shutil.rmtree(temp_dir)
    """
    temp_dir = Path(tempfile.mkdtemp(prefix=f"{prefix}{fold_idx}_"))
    train_dir = temp_dir / "train"
    test_dir = temp_dir / "test"
    train_dir.mkdir()
    test_dir.mkdir()

    test_files = folds[fold_idx]
    train_files = [f for i, fold in enumerate(folds) if i != fold_idx for f in fold]

    for f in train_files:
        shutil.copy2(f, train_dir / f.name)
    for f in test_files:
        shutil.copy2(f, test_dir / f.name)

    print(f"Fold {fold_idx}: train={len(train_files)}, test={len(test_files)}")
    return temp_dir, train_dir, test_dir

In [ ]:
base_dir = "/gscratch/cheme/nlsschim/data/cross_species/tifs/specific_studies/split_adult_wm_gm"
#groups = ["human_female", "human_male", "mice_female", "mice_male", "pig_female", "rat_female", "rat_male"]
groups = ["mice_female", "mice_male", "rat_female", "rat_male"]
groups = ["human_female", "human_male"]#, "pig_female"]

treatment_conditions = ["cortex", "corpus_col"]

k = 3

folds = create_stratified_kfolds(
    base_dir=base_dir,
    groups=groups,
    treatment_conditions=treatment_conditions,
    k=k,
    random_seed=None
)


In [ ]:
# ── K-fold cross-validation loop ───────────────────────────────────────────────

all_predictions = []
all_centroids = []

import vampire

for fold_idx in range(len(folds)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1}/{k}")
    print(f"{'='*60}")

    test_files = folds[fold_idx]
    train_files = [f for j, fold in enumerate(folds) if j != fold_idx for f in fold]

    print(f"Train: {len(train_files)}, Test: {len(test_files)}")

    # Create temp directories
    temp_dir = Path(tempfile.mkdtemp(prefix=f'vampire_fold{fold_idx}_'))
    train_dir = temp_dir / "train"
    test_dir = temp_dir / "test"
    train_dir.mkdir()
    test_dir.mkdir()

    # Copy files
    for f in train_files:
        shutil.copy2(f, train_dir / f.name)
    for f in test_files:
        shutil.copy2(f, test_dir / f.name)

    try:
        # Extract features
        print("Extracting features...")
        vampire.extraction.extract_properties(str(train_dir))

        # Train model
        print("Training model...")
        build_info_df = pd.DataFrame({
            'img_set_path': [str(train_dir)],
            'output_path': [str(train_dir)],
            'model_name': [f'model_fold{fold_idx}'],
            'num_points': [50],
            'num_clusters': [5],
            'num_pc': [np.nan]
        })
        vampire.quickstart.fit_models(build_info_df)

        # Find model file and load centroids
        model_files = [f for f in train_dir.glob("*.pickle") if "model_" in f.name]
        if not model_files:
            raise FileNotFoundError(f"No model .pickle found in {train_dir}")
        model_path = model_files[0]

        with open(model_path, 'rb') as f:
            model = pickle.load(f)

        centroids = model.centroids
        print(f"Centroids shape: {centroids.shape}")
        all_centroids.append(centroids)

        # Apply to test set
        print("Applying to test set...")
        apply_info_df = pd.DataFrame({
            'img_set_path': [str(test_dir)],
            'model_path': [str(model_path)],
            'output_path': [str(test_dir)],
            'img_set_name': [f'fold{fold_idx}']
        })
        vampire.quickstart.transform_datasets(apply_info_df)#, savefig=False)

        # Load predictions
        pred_files = list(test_dir.glob("*apply*.csv"))
        if not pred_files:
            raise FileNotFoundError(f"No predictions CSV found in {test_dir}")

        pred_df = pd.read_csv(pred_files[0])
        pred_df['fold'] = fold_idx
        pred_df['original_shape_mode'] = pred_df['cluster_id']



            # Silhouette score on the fly
        feature_cols = [f for f in morphology_features if f in pred_df.columns]
        if len(feature_cols) >= 2 and pred_df['cluster_id'].nunique() >= 2:
            X_scaled = StandardScaler().fit_transform(pred_df[feature_cols].dropna())
            clean_labels = pred_df.loc[pred_df[feature_cols].dropna().index, 'cluster_id']
            fold_sil_score = silhouette_score(X_scaled, clean_labels)
            pred_df['silhouette_fold_score'] = fold_sil_score  # store on df for later aggregation
            print(f"✓ Collected {len(pred_df)} predictions | Silhouette: {fold_sil_score:.3f}")
        else:
            print(f"✓ Collected {len(pred_df)} predictions | Silhouette: skipped (insufficient clusters/features)")



        all_predictions.append(pred_df)
        print(f"✓ Collected {len(pred_df)} predictions")

    except Exception as e:
        print(f"[ERROR] Fold {fold_idx} failed: {e}")
        raise

    finally:
        shutil.rmtree(temp_dir)

In [ ]:
from pathlib import Path

base_path = Path("/gscratch/cheme/nlsschim/data/cross_species/tifs/specific_studies/adult_wm_gm/")

# Find ALL .tif files recursively
all_files = list(base_path.rglob("*.tif"))

print(f"Total .tif files found: {len(all_files)}")
print(f"\nFirst 5 files:")
for f in all_files[:5]:
    print(f"  {f.relative_to(base_path)}")

print(f"\nLast 5 files:")
for f in all_files[-5:]:
    print(f"  {f.relative_to(base_path)}")

In [ ]:
from pathlib import Path

base_path = Path("/gscratch/cheme/nlsschim/data/cross_species/tifs/specific_studies/adult_wm_gm/")
all_files = list(base_path.rglob("*.tif"))

# Each file is its own "group"
groups = {f.stem: [f] for f in all_files}

print(f"Total files (groups): {len(groups)}")
print(f"\nFirst 5:")
for i, (group_id, files) in enumerate(list(groups.items())[:5]):
    print(f"  {group_id}")

In [ ]:
from pathlib import Path
import random

base_path = Path("/gscratch/cheme/nlsschim/data/cross_species/tifs/specific_studies/adult_wm_gm/")
all_files = list(base_path.rglob("*.tif"))

# Shuffle files for random splits
random.seed(42)
shuffled_files = all_files.copy()
random.shuffle(shuffled_files)

# Split into 5 folds
k = 5
fold_size = len(shuffled_files) // k
folds = []

for i in range(k):
    start = i * fold_size
    end = start + fold_size if i < k - 1 else len(shuffled_files)
    folds.append(shuffled_files[start:end])

print(f"Total files: {len(shuffled_files)}")
print(f"Number of folds: {len(folds)}")
print(f"Fold sizes: {[len(f) for f in folds]}")

# Show what fold 0 looks like
print(f"\nFold 0 (first 3 files):")
for f in folds[0][:3]:
    print(f"  {f.name}")

# Create train/test splits
print(f"\nTrain/test splits:")
for i in range(k):
    test = folds[i]
    train = [f for j, fold in enumerate(folds) if j != i for f in fold]
    print(f"Fold {i}: {len(train)} train, {len(test)} test")

In [ ]:
from pathlib import Path
import random
import shutil
import tempfile
import pandas as pd
import numpy as np
import pickle
from scipy.optimize import linear_sum_assignment

base_path = Path("/gscratch/cheme/nlsschim/data/cross_species/tifs/specific_studies/adult_wm_gm/")
all_files = list(base_path.rglob("*.tif"))

# Shuffle and split
random.seed(42)
shuffled_files = all_files.copy()
random.shuffle(shuffled_files)

k = 5
fold_size = len(shuffled_files) // k
folds = []
for i in range(k):
    start = i * fold_size
    end = start + fold_size if i < k - 1 else len(shuffled_files)
    folds.append(shuffled_files[start:end])

print(f"Total files: {len(all_files)}")
print(f"Fold sizes: {[len(f) for f in folds]}")

# Store predictions AND centroids
all_predictions = []
all_centroids = []

# Loop through all folds
for fold_idx in range(k):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1}/{k}")
    print(f"{'='*60}")
    
    test_files = folds[fold_idx]
    train_files = [f for j, fold in enumerate(folds) if j != fold_idx for f in fold]
    
    print(f"Train: {len(train_files)}, Test: {len(test_files)}")
    
    # Create temp directories
    temp_dir = Path(tempfile.mkdtemp(prefix=f'vampire_fold{fold_idx}_'))
    train_dir = temp_dir / "train"
    test_dir = temp_dir / "test"
    train_dir.mkdir()
    test_dir.mkdir()
    
    # Copy files
    for f in train_files:
        shutil.copy2(f, train_dir / f.name)
    for f in test_files:
        shutil.copy2(f, test_dir / f.name)
    
    # Extract features
    print("Extracting features...")
    import vampire
    vampire.extraction.extract_properties(str(train_dir))
    
    # Train model
    print("Training model...")
    build_info_df = pd.DataFrame({
        'img_set_path': [str(train_dir)],
        'output_path': [str(train_dir)],
        'model_name': [f'model_fold{fold_idx}'],
        'num_points': [50],
        'num_clusters': [5],
        'num_pc': [np.nan]
    })
    vampire.quickstart.fit_models(build_info_df)
    
    # Find model and load centroids
    model_files = [f for f in train_dir.glob("*.pickle") if "model_" in f.name]
    model_path = model_files[0]
    
    # Load the model to get centroids
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    
    centroids = model.centroids
    print(f"Centroids shape: {centroids.shape}")
    all_centroids.append(centroids)
    
    # Apply to test set
    print("Applying to test set...")
    apply_info_df = pd.DataFrame({
        'img_set_path': [str(test_dir)],
        'model_path': [str(model_path)],
        'output_path': [str(test_dir)],
        'img_set_name': [f'fold{fold_idx}']
    })
    vampire.quickstart.transform_datasets(apply_info_df)
    
    # Load predictions
    pred_files = list(test_dir.glob("*apply*.csv"))
    if pred_files:
        pred_df = pd.read_csv(pred_files[0])
        pred_df['fold'] = fold_idx
        pred_df['original_shape_mode'] = pred_df['cluster_id']  # Keep original
        all_predictions.append(pred_df)
        print(f"✓ Collected {len(pred_df)} predictions")
    
    # Cleanup
    shutil.rmtree(temp_dir)

print(f"\n{'='*60}")
print("ALIGNING SHAPE MODES")
print(f"{'='*60}")

# Use fold 0 as reference
reference_centroids = all_centroids[0]
print(f"\nUsing fold 0 as reference")
print(f"Reference centroids shape: {reference_centroids.shape}")

# Align each fold to the reference
for fold_idx in range(k):
    if fold_idx == 0:
        print(f"Fold 0: Reference (no alignment needed)")
        continue
    
    fold_centroids = all_centroids[fold_idx]
    
    # Compute pairwise distances between reference and fold centroids
    distances = np.zeros((reference_centroids.shape[0], fold_centroids.shape[0]))
    for i, ref_centroid in enumerate(reference_centroids):
        for j, fold_centroid in enumerate(fold_centroids):
            distances[i, j] = np.linalg.norm(ref_centroid - fold_centroid)
    
    # Hungarian algorithm to find best matching
    row_ind, col_ind = linear_sum_assignment(distances)
    
    # Create mapping: old_mode -> new_mode
    mapping = {col_ind[i]: row_ind[i] for i in range(len(row_ind))}
    print(f"Fold {fold_idx}: Mapping {mapping}")
    
    # Remap shape modes in predictions
    pred_df = all_predictions[fold_idx]
    pred_df['shape_mode'] = pred_df['original_shape_mode'].map(mapping)

# Combine all predictions
print(f"\n{'='*60}")
print("COMBINING RESULTS")
print(f"{'='*60}")

final_predictions = pd.concat(all_predictions, ignore_index=True)

print(f"\nTotal predictions: {len(final_predictions)}")
print(f"Aligned shape mode distribution:")
print(final_predictions['shape_mode'].value_counts().sort_index())

# Save results
output_file = "cross_species_kfold_aligned_predictions.csv"
final_predictions.to_csv(output_file, index=False)
print(f"\n✓ Saved to: {output_file}")

print("\nDone! 🎉")

In [ ]:
final_predictions

In [ ]:
import pandas as pd
import numpy as np

# Load the predictions
df = pd.read_csv("cross_species_kfold_aligned_predictions.csv")

print("="*60)
print("K-FOLD CROSS-VALIDATION RESULTS")
print("="*60)

print(f"\nTotal predictions: {len(df)}")
print(f"Expected: 257 files")
print(f"Missing: {257 - len(df)} (probably failed processing)")

print("\n--- Aligned Shape Mode Distribution ---")
print(df['shape_mode'].value_counts().sort_index())

print("\n--- Original vs Aligned Comparison ---")
for fold in range(5):
    fold_df = df[df['fold'] == fold]
    print(f"\nFold {fold}:")
    print(f"  Original: {fold_df['original_shape_mode'].value_counts().sort_index().to_dict()}")
    print(f"  Aligned:  {fold_df['shape_mode'].value_counts().sort_index().to_dict()}")

# Extract species from filename
def extract_species(filename):
    name = filename.lower()
    if 'mouse' in name or 'ms' in name or 'fs' in name:
        return 'mouse'
    elif 'rat' in name:
        return 'rat'
    elif 'uwa' in name:
        return 'human'
    elif 'pig' in name or 'sow' in name:
        return 'pig'
    else:
        return 'unknown'

# Extract region from filename
def extract_region(filename):
    name = filename.lower()
    if 'corpus' in name or '_cc' in name:
        return 'corpus_callosum'
    elif 'cortex' in name or 'cor' in name:
        return 'cortex'
    else:
        return 'unknown'

# Add metadata
df['species'] = df['filename'].apply(extract_species)
df['region'] = df['filename'].apply(extract_region)

print("\n--- Shape Modes by Species ---")
species_modes = df.groupby(['species', 'shape_mode']).size().unstack(fill_value=0)
print(species_modes)

print("\n--- Shape Modes by Region ---")
region_modes = df.groupby(['region', 'shape_mode']).size().unstack(fill_value=0)
print(region_modes)

# Save enhanced version
df.to_csv("cross_species_kfold_aligned_predictions_enhanced.csv", index=False)
print("\n✓ Saved enhanced version with species/region metadata")